# Step 2 · Planarian and human model training

Each step presents the planarian analysis first, followed by the separate human analysis.


In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
ROOT=next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p/'dataset').is_dir() and (p/'results').is_dir()), None)
assert ROOT is not None, 'Start Jupyter within the repository'
assert (ROOT/'dataset').is_dir() and (ROOT/'results').is_dir(), 'Start Jupyter from the repository root or notebooks/'
sys.path.insert(0,str(ROOT))
PROJECT=ROOT/'dataset'
PANDO=ROOT/'results/pando'
MODEL=ROOT/'results/model'
SCENIC=ROOT/'results/scenic'
SCENIC_DATA=ROOT/'dataset/scenic'
pd.set_option('display.max_columns',30)

print('Package:',ROOT)
print('Python:',sys.version.split()[0])
print('Directories available:', all(p.exists() for p in [PROJECT,PANDO,MODEL,SCENIC]))

Package: .
Python: 3.11.18
Directories available: True


## Planarian · three-stage model outputs

The planarian workflow first creates gene embeddings, then fits RNA-to-embedding and seven-class edge models. Below are outputs from the available trained planarian checkpoints; this cell does not retrain them. Training from scratch requires the planarian RNA PCA input and a compatible legacy environment; see the separate planarian tutorial.


In [2]:
planarian_result=ROOT/'results/planarian'
with np.load(planarian_result/'phase1_output/gene_z.npz',allow_pickle=False) as data:
 print('Planarian phase 1 embeddings:',data['z'].shape)
with np.load(planarian_result/'phase2_output/all_gene_z.npz',allow_pickle=False) as data:
 print('Planarian phase 2 embeddings:',data['z'].shape)
print('Phase 2 trained model:',(planarian_result/'phase2_output/rna2z_model.pt').is_file())
print('Phase 3 trained model:',(planarian_result/'phase3_output/edge_predictor.pt').is_file())


Planarian phase 1 embeddings: (1642, 64)
Planarian phase 2 embeddings: (50106, 64)
Phase 2 trained model: True
Phase 3 trained model: True


## Human · model training and validation

The current model is a binary neural edge scorer, not the historical seven-class classifier. It uses the processed Pando/RNA/motif/GRNBoost2 features from Step 1. TFs, not individual edges, are split into training, validation and test groups; GLI3 outgoing edges are excluded when constructing training-network features.

In [3]:
manifest=json.loads((MODEL/'manifest.json').read_text())
print(json.dumps(manifest,indent=2))
metrics=pd.read_csv(MODEL/'validation_metrics.tsv',sep='\t')
selected=manifest['selected_model']
print(metrics[metrics.method.eq(selected)][['method','split','TFs','mean_AP','mean_hits_top10']].to_string(index=False))

{
  "seed": 42,
  "common_genes": 1794,
  "benchmark_positive": 48,
  "train_TFs": 306,
  "validation_TFs": 49,
  "test_TFs": 49,
  "train_edges": 5018,
  "selected_model": "GRNM_32_drop0.1",
  "selected_epoch": 20,
  "selection": "mean per-TF AP on validation TFs; architecture & epoch chosen without GLI3 candidate table",
  "negative_sampling": "four random unlisted pairs per known Pando edge, not verified negatives",
  "train_features": "target degree excluding queried edge; RNA correlation; GRNBoost2 importance; motif similarity; TF and target in-degree",
  "heldout_features": "all outgoing edges of validation TFs, test TFs and GLI3 excluded before feature construction",
  "seed_rank_note": "Pando seed only reranking, full scores have no significance or motif-pruning cutoff"
}
         method      split  TFs  mean_AP  mean_hits_top10
GRNM_32_drop0.1 validation   49 0.258688         3.346939
GRNM_32_drop0.1       test   49 0.285040         3.551020


The held-out test measures recovery of **Pando-inferred edges**, not independent experimental truth. Unlisted pairs were sampled as training background, not verified biological negatives. The original seven-class model was not tested under this same TF split.

## Optional retraining

The default tutorial reads the saved model. To train a separate copy, install PyTorch and set `RUN_MODEL_TRAINING=1`. The cell creates a new uniquely named folder under `results/generated/` and will not overwrite the archived model. It uses the existing input files and can take several minutes. This is an experimental rerun; exact reproduction depends on package versions.

In [4]:
import os, subprocess, datetime
if os.environ.get('RUN_MODEL_TRAINING')=='1':
    run=ROOT/'results/generated'/('training_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    print('Training into',run)
    subprocess.run([sys.executable,str(ROOT/'src/model/train.py'),'--out',str(run)],check=True,cwd=ROOT)
else:
    print('Training skipped; set RUN_MODEL_TRAINING=1 to train in a new directory.')

Training skipped; set RUN_MODEL_TRAINING=1 to train in a new directory.


## Reconstruct scores using saved weights

If `dataset/scenic/expression.mtx` is available, this cell reconstructs the scores from the saved model. The matrix is excluded from the upload; without it, the cell displays the saved ranking and explicitly skips reconstruction. Retraining also requires this matrix. See the README for data availability.


In [5]:
saved=pd.read_csv(MODEL/'GLI3_all_scores.tsv',sep='\t')
if (SCENIC_DATA/'expression.mtx').exists():
 from src.grnm_review import reconstruct_gli3_scores
 reconstructed=reconstruct_gli3_scores()
 print(reconstructed.head(10).to_string(index=False))
else:
 print('Expression matrix not included: showing saved rankings; score reconstruction skipped. See README for data availability.')
 print(saved.head(10).to_string(index=False))


Expression matrix not included: showing saved rankings; score reconstruction skipped. See README for data availability.
target    score
  HES5 0.942408
  PAX6 0.937604
  LHX2 0.923951
 DACH1 0.923514
  HES4 0.923230
   ID4 0.922542
ZBTB16 0.919254
  SOX9 0.901088
 FOXG1 0.898161
 NR2F1 0.894505


HES4 was explicitly added to the candidate universe after prior inspection and the model was retrained. This makes HES4/HES5 analysis exploratory, despite the ranking itself being computed by the model rather than manually edited. See `docs/publication_scope.md`.